# 匯入套件與設定基礎變數

In [28]:
import os
import requests
import pandas as pd
from datetime import datetime

# 設定目標股票
stock_ids = ["2881", "2882", "2883", "2884", "2885", "2887", "2890", "2891", "2892"]
base_dir = "data"

# 建立自動化資料夾結構

In [29]:
# 確保主資料夾與各股票專屬的子資料夾都存在
for stock_id in stock_ids:
    folder_path = os.path.join(base_dir, stock_id)
    os.makedirs(folder_path, exist_ok=True)
    
print("資料夾結構檢查與建立完成！")

資料夾結構檢查與建立完成！


# 向證交所抓取資料並分發存檔

In [30]:
def get_real_latest_trading_date():
    """向官方大盤 API 查詢最近一個真實的交易日期"""
    url = "https://openapi.twse.com.tw/v1/exchangeReport/FMTQIK"
    try:
        resp = requests.get(url)
        resp.raise_for_status()
        data = resp.json()
        
        # 抓取資料庫中最後一筆（最新）的大盤紀錄
        latest_record = data[-1]
        roc_date_str = latest_record['Date'] # 格式如 "1130223" (民國年)
        
        # 將民國年轉為西元年 (YYYY-MM-DD)
        year = int(roc_date_str[:-4]) + 1911
        month = roc_date_str[-4:-2]
        day = roc_date_str[-2:]
        real_date = f"{year}-{month}-{day}"
        
        return real_date
    except Exception as e:
        print(f"抓取大盤日期失敗: {e}")
        # 備用防呆：萬一連線失敗，退回當天日期
        return datetime.today().strftime('%Y-%m-%d')

def fetch_and_save_price_perfect(stock_ids, base_dir):
    # 1. 🌟 先去查明真正的「最後交易日」是哪一天
    trading_date = get_real_latest_trading_date()
    print(f"📢 經過大盤 API 確認，目前官方最新的真實交易日為：{trading_date}")
    
    # 2. 再去抓取所有股票的收盤價
    url = "https://openapi.twse.com.tw/v1/exchangeReport/STOCK_DAY_ALL"
    
    try:
        print("正在向證交所請求最新股價資料...")
        resp = requests.get(url)
        resp.raise_for_status()
        
        data = resp.json()
        df = pd.DataFrame(data)
        df_target = df[df['Code'].isin(stock_ids)].copy()
        
        # 填上我們剛剛查到的 100% 準確日期
        df_target['Date'] = trading_date
        df_target = df_target[['Date', 'Code', 'Name', 'ClosingPrice']]
        df_target.rename(columns={'Code': 'stock_id', 'Name': 'name', 'ClosingPrice': 'price'}, inplace=True)
        
        for index, row in df_target.iterrows():
            sid = row['stock_id']
            file_path = os.path.join(base_dir, sid, f"{sid}_price.csv")
            
            current_price = float(row['price']) if row['price'] else 0.0
            
            df_save = pd.DataFrame([{
                'Date': row['Date'],
                'stock_id': sid,
                'name': row['name'],
                'price': current_price
            }])
            
            if os.path.exists(file_path):
                df_old = pd.read_csv(file_path)
                df_final = pd.concat([df_old, df_save], ignore_index=True)
                # 以正確的日期進行去重複
                df_final = df_final.drop_duplicates(subset=['Date'], keep='last')
            else:
                df_final = df_save
                
            df_final.to_csv(file_path, index=False)
            
        print("✅ 價格存檔完成")
            
    except Exception as e:
        print(f"❌ 抓取或存檔過程發生錯誤: {e}")

# 執行看看
fetch_and_save_price_perfect(stock_ids, base_dir)

📢 經過大盤 API 確認，目前官方最新的真實交易日為：2026-02-23
正在向證交所請求最新股價資料...
✅ 價格存檔完成


# 抓取官方估值資料

In [31]:
def safe_float(val):
    try:
        if not val or str(val).strip() in ["", "-"]:
            return 0.0
        return float(val)
    except ValueError:
        return 0.0

def update_valuation_data_final(stock_ids, base_dir):
    # 取得真實交易日 (沿用之前的函數)
    trading_date = get_real_latest_trading_date()
    url = "https://openapi.twse.com.tw/v1/exchangeReport/BWIBBU_ALL"
    
    try:
        print(f"📢 準備更新估值資料，目標日期為：{trading_date}")
        resp = requests.get(url)
        resp.raise_for_status()
        
        df = pd.DataFrame(resp.json())
        df_target = df[df['Code'].isin(stock_ids)].copy()
        
        for index, row in df_target.iterrows():
            sid = row['Code']
            file_path = os.path.join(base_dir, sid, f"{sid}_valuation.csv")
            
            # 移除不存在的 DividendYear，只保留我們需要的核心數據
            new_row = pd.DataFrame([{
                'Date': trading_date,
                'stock_id': sid,
                'name': row.get('Name', ''),
                'Yield(%)': safe_float(row.get('DividendYield')),  # 殖利率
                'PE_Ratio': safe_float(row.get('PEratio')),        # 本益比
                'PB_Ratio': safe_float(row.get('PBratio'))         # 股價淨值比
            }])
            
            if os.path.exists(file_path):
                df_old = pd.read_csv(file_path)
                df_final = pd.concat([df_old, new_row], ignore_index=True)
                df_final = df_final.drop_duplicates(subset=['Date'], keep='last')
            else:
                df_final = new_row
                
            df_final.to_csv(file_path, index=False)
            
        print("✅ 殖利率與估值資料存檔完成！")
        
    except Exception as e:
        print(f"❌ 發生錯誤: {e}")

# 執行更新
update_valuation_data_final(stock_ids, base_dir)

📢 準備更新估值資料，目標日期為：2026-02-23
✅ 殖利率與估值資料存檔完成！
